# Sensitive or deployment-related content removed.


In [ ]:
# Sensitive or deployment-related content removed.


# Importing **Python Libraries and Models**

In [ ]:
%%writefile backend.py
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# **Loading Dataset**

In [ ]:
print("Loading Dataset...")
data = "heart.csv"
print("Heart Dataset successfully loaded")


## Exploratory Data Analysis

In [ ]:
import pandas as pd
data = pd.read_csv("heart.csv")
print("\n===== BASIC INFO =====")
print("\nMissing Values:\n", data.isnull().sum())
print("\nClass Distribution:\n", data["target"].value_counts())
print("\nCorrelation with Target:\n")
print(data.corr()["target"].sort_values(ascending=False))

Feature and Target Splitting

In [ ]:
X = data.drop("target", axis=1)
y = data["target"]
df = pd.read_csv("heart.csv")
print(df.columns)

Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Standard Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Data Modelling

In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pickle

models = {
    "SVM": SVC(probability=True),
    "Random Forest": RandomForestClassifier(),
    "Decision Tree": DecisionTreeClassifier(),
    "Logistic Regression": LogisticRegression(),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric="logloss")

}

results = {}

print("\n===== MODEL EVALUATION =====\n")

for name, model in models.items():

    if name == "SVM":
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    results[name] = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "model": model
    }

    print(name)
    print("Accuracy :", round(results[name]["accuracy"], 3))
    print("Precision:", round(results[name]["precision"], 3))
    print("Recall   :", round(results[name]["recall"], 3))
    print("F1 Score :", round(results[name]["f1"], 3))
    print("-----------------------------")

best_model_name = max(results, key=lambda x: results[x]["recall"])
best_model = results[best_model_name]["model"]

print("\nBest Model Based on Highest Recall:", best_model_name)

pickle.dump(best_model, open("best_heart_model.pkl", "wb"))
pickle.dump(scaler, open("scaler.pkl", "wb"))

print("\nModel and scaler saved successfully.")

Running backend python code

In [ ]:
!python backend.py

# Front-End code : **App.py**

In [ ]:
import streamlit as st

In [ ]:
pip install st-annotated-text

In [ ]:
%%writefile app.py
import numpy as np
import pickle
import streamlit as st
import datetime
from annotated_text import annotated_text
from PIL import Image
img= Image.open("logo.jpg")
st.markdown(
    """
    <style>
    .block-container {
        padding-top: 0rem;
        padding-bottom: 0rem;
    }

    header[data-testid="stHeader"] {
        background-color: rgba(0,0,0,0);
    }

    .stApp {
        background: linear-gradient(180deg, #e3f2fd 0%, #ffffff 40%);
        background-attachment: fixed;
    }

    .main .block-container {
        padding-top: 2rem;
    }
    </style>
    """,
    unsafe_allow_html=True
)
st.set_page_config(page_title="Multi Disease Risk Prediction", layout="wide", initial_sidebar_state="expanded")
st.markdown("""
    <style>
    .block-container {
        padding-top: 2rem;
        padding-bottom: 2rem;
        padding-left: 5rem;
        padding-right: 5rem;
    }
    </style>
    """, unsafe_allow_html=True)

if "registered" not in st.session_state:
    st.session_state.registered = False

if not st.session_state.registered:   #Registration Page

    top_col1, top_col2 = st.columns([3, 1])

    with top_col1:
        st.title("📋 Patient Registration")
        st.write("Please enter your details before proceeding to the prediction system.")

    with top_col2:
        patient_photo = st.camera_input("Take Patient Photo", label_visibility="collapsed")

    st.markdown("---")

    col1, col2, col3 = st.columns(3)

    with col1:
        patient_id = st.text_input("Patient ID *", "19783")
        full_name = st.text_input("Full Name *", "Yash Pandey")
    with col2:
        age = st.number_input("Age", 1, 120, 25)
        gender = st.selectbox("Gender", ["Male", "Female"])
    with col3:
        contact = st.text_input("Contact Number *", "1234567893")
        visit_date = st.date_input("Date of Visit", datetime.date.today())

    if st.button("Proceed to Prediction →"):

        st.session_state.patient_id = patient_id
        st.session_state.full_name = full_name
        st.session_state.age = age
        st.session_state.gender = gender
        st.session_state.visit_date = visit_date

        st.session_state.registered = True
        st.rerun()

else:

    st.title("🩺 Multi Disease Risk Prediction System")

    with st.container():
        annotated_text(
            (st.session_state.full_name, "Patient", "#8ef"),
            " ",
            (f"ID: {st.session_state.patient_id}", "System ID", "#faa"),
            " ",
            (f"{st.session_state.age} Years", "Age", "#fea"),
            " ",
            (st.session_state.gender, "Gender", "#afa"),
            " ",
            (str(st.session_state.visit_date), "Visit Date", "#eee"),
        )
    model = pickle.load(open("best_heart_model.pkl", "rb"))
    scaler = pickle.load(open("scaler.pkl", "rb"))

    disease_type = st.radio(
        "Select Disease",
        ["Heart Disease", "Diabetes", "Kidney Disease"],
        horizontal=True
    )

    st.markdown("---")

    if disease_type == "Heart Disease":

        st.subheader("❤️ Heart Disease Prediction")
        st.info("Valid Age Range: 16 - 90 Years")

        col1, col2 ,col3 = st.columns(3)

        with col1:
            age = st.number_input("Age", 16, 90, 30)
            gender = st.selectbox("Gender", ["Male", "Female"])
            chest_pain_help = """
            **Identify your chest pain:**
            * **Typical Angina:** Squeezing or heavy pressure during activity.
            * **Atypical Angina:** Sharp poking or shortness of breath.
            * **Non-anginal:** Muscle-like pain or pain when breathing deeply.
            * **Asymptomatic:** No pain felt at all.
            """
            chest_pain = st.selectbox(
                "Chest Pain Type",
                ["Typical Angina", "Atypical Angina",
                 "Non-anginal Pain", "Asymptomatic"],help=chest_pain_help
            )
            resting_bp = st.number_input("Resting BP", 80, 250, 120)

        with col2:
            cholesterol = st.number_input("Cholesterol", 100, 600, 200)
            fasting_bs = st.selectbox("Fasting Blood Sugar > 120 mg/dl", ["True", "False"])
            rest_ecg = st.selectbox(
                "Rest ECG",
                ["Normal", "ST-T Abnormality", "LV Hypertrophy"]
            )
            max_hr = st.number_input("Max Heart Rate", 60, 220, 150)
        with col3:
            exang = st.selectbox("Exercise Induced Angina", ["Yes", "No"], help="Do you experience chest pain specifically during physical exercise?")
            oldpeak = st.number_input("Oldpeak", 0.0, 10.0, 1.0)
            slope = st.selectbox("ST Slope", ["Upsloping", "Flat", "Downsloping"])
            ca = st.number_input("Number of Major Vessels (0-4)", 0, 4, 0)
            thal_help = """
            **Based on your Stress Test (Thallium Scan):**
            * **Normal:** Your doctor said your heart looks healthy and clear.
            * **Fixed Defect:** You have some permanent scarring (often from a previous heart attack).
            * **Reversible Defect:** Your heart gets enough blood at rest, but struggles to get enough blood during exercise.
            """
            thal = st.selectbox(
                "Thalassemia",
                ["Normal", "Fixed Defect", "Reversible Defect"], help= thal_help
            )

        if st.button("Predict"):

            gender = 1 if gender == "Male" else 0
            chest_pain = ["Typical Angina", "Atypical Angina",
                          "Non-anginal Pain", "Asymptomatic"].index(chest_pain)
            fasting_bs = 1 if fasting_bs == "True" else 0
            rest_ecg = ["Normal", "ST-T Abnormality", "LV Hypertrophy"].index(rest_ecg)
            exang = 1 if exang == "Yes" else 0
            slope = ["Upsloping", "Flat", "Downsloping"].index(slope)
            thal = ["Normal", "Fixed Defect", "Reversible Defect"].index(thal)

            input_data = np.array([[age, gender, chest_pain,
                                    resting_bp, cholesterol,
                                    fasting_bs, rest_ecg,
                                    max_hr, exang, oldpeak,
                                    slope, ca, thal]])

            input_scaled = scaler.transform(input_data)
            result = model.predict(input_scaled)

            if result[0] == 1:
                st.error("⚠ High Risk of Heart Disease")
            else:
                st.success("✅ Low Risk of Heart Disease")

    elif disease_type == "Diabetes":
        st.subheader("🩸 Diabetes Disease Prediction")
        st.info("Valid Age Range: 0 - 80 Years")

        col1, col2= st.columns(2)

        with col1:
            gender = st.selectbox("Gender", ["Female", "Male", "Other"])
            age = st.number_input("Age", 0, 80, 30)
            hypertension = st.selectbox("Hypertension", ["No", "Yes"])
            heart_disease = st.selectbox("Heart Disease", ["No", "Yes"])

        with col2:
            smoking_help = """
            **Please select the option that best fits:**
            * **Never:** You have never smoked or have smoked fewer than 100 cigarettes in your entire life.
            * **Current:** You currently smoke every day or some days.
            * **Former:** You used to smoke regularly but have completely quit.
            * **Ever:** You have a history of smoking (at least 100 cigarettes), but it doesn't specify if you stopped.
            * **Not Current:** You have smoked in the past, but you do not smoke right now.
            * **No Info:** Choose this if you are unsure or do not wish to say.
            """
            smoking_history = st.selectbox(
                "Smoking History",
                ["never", "No Info", "current", "former", "ever", "not current"], help=smoking_help
            )
            bmi = st.number_input("BMI", 10.0, 95.0, 25.0)
            hba1c_level = st.number_input("HbA1c Level", 3.5, 9.0, 5.5)
            blood_glucose = st.number_input("Blood Glucose Level", 80, 300, 100)


        if st.button("Predict"):
            st.success("Backend integration pending")



    elif disease_type == "Kidney Disease":
        st.subheader("🧪 Chronic Kidney Disease Prediction")
        st.info("Enter clinical laboratory values to assess kidney function.")

        col1, col2, col3, col4 = st.columns(4)

        with col1:
            bp = st.number_input("Blood Pressure (mm/Hg)", 50, 200, 80)
            sg = st.selectbox("Specific Gravity", [1.005, 1.010, 1.015, 1.020, 1.025])
            al = st.selectbox("Albumin (0-5)", [0, 1, 2, 3, 4, 5])
            su = st.selectbox("Sugar (0-5)", [0, 1, 2, 3, 4, 5])
        with col2:
            rbc = st.selectbox("Red Blood Cells", ["normal", "abnormal"])
            pc = st.selectbox("Pus Cell", ["normal", "abnormal"])
            pcc = st.selectbox("Pus Cell Clumps", ["notpresent", "present"])
            ba_help = """
            **Check your lab report for "Urine Culture" or "Bacteria":**
            * **Not Present:** No significant bacterial growth found.
            * **Present:** Bacteria like *E. coli*, *Klebsiella*, or *Proteus* were detected.
            **Note:** This usually indicates an infection (UTI) that can cause temporary or permanent kidney stress.
            """
            ba = st.selectbox("Bacteria", ["not present", "present"],help=ba_help)

        with col3:
            bgr = st.number_input("Blood Glucose Random (mgs/dl)", 20, 500, 120)
            bu = st.number_input("Blood Urea (mgs/dl)", 1, 400, 40)
            sc = st.number_input("Serum Creatinine (mgs/dl)", 0.0, 15.0, 1.2)
            sod = st.number_input("Sodium (mEq/L)", 100, 170, 135)
        with col4:
            pot = st.number_input("Potassium (mEq/L)", 2.0, 8.0, 4.5)
            hemo = st.number_input("Hemoglobin (gms)", 3.0, 20.0, 13.0)
            htn = st.selectbox("Hypertension", ["no", "yes"])
            dm = st.selectbox("Diabetes Mellitus", ["no", "yes"])

        if st.button("Predict Kidney Health"):
            # Feature Processing (matching UCI Dataset encoding)
            rbc = 1 if rbc == "normal" else 0
            pc = 1 if pc == "normal" else 0
            pcc = 1 if pcc == "present" else 0
            ba = 1 if ba == "present" else 0
            htn = 1 if htn == "yes" else 0
            dm = 1 if dm == "yes" else 0
            features = np.array([[bp, sg, al, su, rbc, pc, pcc, ba, bgr, bu, sc, sod, pot, hemo, htn, dm]])

            st.success("Backend Integration Pending")
    st.markdown("---")
    st.markdown(
        "<div style='text-align:center;'>"
        "<b>Capstone Project 2026 ( Btech CSE )</b><br>"
        "DIT UNIVERSITY , Dehradun<br>"
        "Developed by Abhyudita Pandey | Nishka Sharma | Raghav Sharma"
        "</div>",
        unsafe_allow_html=True
    )

# Running Streamlit..

In [ ]:
# Sensitive or deployment-related content removed.


# Sensitive or deployment-related content removed.


In [ ]:
# Sensitive or deployment-related content removed.
